# GRU Model Multivariate


In this section we implement multivariate forecasting using the GRU Model (Gated Recurrent Unit) with the **TimeSeriesDatasetVectorizedExog** approach.

The GRU (Gated Recurrent Unit) Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, **TimeSeriesDatasetVectorizedExog** batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate).

The model architecture remains unchanged, we simply reshape the data to process all series in parallel, achieving faster training while incorporating exogenous variables. The model balances the complexity of LSTMs with the simplicity of vanilla RNNs, using two gates (reset and update) to control information flow.

**Layer Breakdown**

- GRU Layers: 2 stacked GRU layers with gating mechanisms
- Hidden Size: 128 units per layer (default)
- Dropout: Applied between GRU layers (if >1 layer) and before final output
- Output Layer: Single fully connected layer producing 1-step forecast

## Model

In [ ]:
import torch
import torch.nn as nn

class GRUForecaster(nn.Module):
    """
    GRU model for MULTIVARIATE time series forecasting.
    Architecture: GRU -> Dropout -> GRU -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    
    GRU is similar to LSTM but with fewer parameters (no cell state).
    Generally faster than LSTM while maintaining good performance.
    Uses reset and update gates instead of LSTM's input/forget/output gates.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: GRU hidden dimension
            num_layers: Number of GRU layers
            dropout: Dropout rate
        """
        super(GRUForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # GRU layers
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # GRU forward pass
        # gru_out: (batch_size, seq_length, hidden_size)
        # h_n: (num_layers, batch_size, hidden_size)
        gru_out, h_n = self.gru(x)
        
        # Take the output from the last time step
        last_output = gru_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out


## Model Results without Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration (s) |
|-------|-----------------|------------|---------|-------------|---------------|------------|--------------|
| 0 | 0.1428 | 8 | 0.1474 | 256 | 0.0001238 | 3 | 41.28 |
| 1 | 0.2144 | 16 | 0.4802 | 64 | 0.0001244 | 2 | 5.13 |
| 2 | 0.1234 | 8 | 0.2748 | 256 | 0.0010085 | 2 | 26.47 |
| 3 | 0.1327 | 8 | 0.2510 | 256 | 0.0002088 | 3 | 41.37 |
| 4 | 0.1360 | 16 | 0.4979 | 256 | 0.0002944 | 2 | 26.69 |


### Best Hyperparameters 

Parameters for Trial 2:
- learning_rate: 0.0010
- batch_size: 8
- num_layers: 2
- hidden_size: 256
- dropout: 0.2748


#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/gru/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/gru/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 3 Results](./img/multivariate/gru/fold3/fold_results.png)


### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE (%) |
|------|----------|----------|----------|--------|-----------|
| Fold 1 | 61047.46 | 247.08 | 106.79 | 0.8790 | 58.79 |
| Fold 2 | 54456.84 | 233.36 | 100.93 | 0.8797 | 58.18 |
| Fold 3 | 51351.28 | 226.61 | 94.93 | 0.8944 | 86.47 |
| **Average** | **55618.53 ± 4951.37** | **235.68 ± 10.43** | **100.89 ± 5.93** | **0.8844 ± 0.0087** | **67.82 ± 16.16** |

### SMAPE Distribution Accross Folds 

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|----------------------|
| <10% | 11.0% ± 7.1% | 165 |
| 10-20% | 11.6% ± 1.2% | 175 |
| 20-30% | 12.6% ± 2.8% | 189 |
| 30-40% | 10.1% ± 1.1% | 151 |
| >40% | 54.7% ± 3.1% | 822 |

**Comparison with Baseline:**

The GRU multivariate model achieves an average SMAPE of 67.82% ± 16.16%, which is **4.44 percentage points lower** than the baseline 3-month rolling average (72.26% ± 7.06%). While the GRU model shows improvement over the baseline, it exhibits higher variation across folds (standard deviation: 16.16% vs 7.06%), primarily due to Fold 3's higher SMAPE (86.47%). Despite this variability, the model demonstrates strong performance in Folds 1 and 2 (58.79% and 58.18% respectively), indicating its ability to capture temporal dependencies efficiently. The GRU's simplified gating mechanism compared to LSTM provides faster training while maintaining competitive forecasting accuracy across most time periods.


## Model Results with Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration (s) |
|-------|-----------------|------------|---------|-------------|---------------|------------|--------------|
| 0 | 0.1918 | 4 | 0.4555 | 32 | 0.0025 | 2 | 4.06 |
| 1 | 0.2596 | 8 | 0.1288 | 64 | 0.0015 | 3 | 5.28 |
| 2 | 0.5340 | 8 | 0.4539 | 32 | 0.0006 | 1 | 0.61 |
| 3 | 0.2928 | 4 | 0.2789 | 64 | 0.0009 | 3 | 4.78 |
| 4 | 0.2363 | 8 | 0.2830 | 32 | 0.0008 | 2 | 3.08 |

### Best Hyperparameters

Parameters for Trial 0:
- learning_rate: 0.00245
- batch_size: 4
- num_layers: 2
- hidden_size: 32
- dropout: 0.455

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/gru_exog/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/gru_exog/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

    
![Fold 3 Results](./img/multivariate/gru_exog/fold3/fold_results.png)


### Fold Results 

| Fold | MSE | RMSE | MAE | R² | SMAPE (%) |
|------|----------|----------|----------|--------|-----------|
| Fold 1 | 222737.09 | 471.95 | 227.61 | 0.5586 | 116.32 |
| Fold 2 | 220260.35 | 469.32 | 225.71 | 0.5136 | 142.46 |
| Fold 3 | 150746.09 | 388.26 | 173.39 | 0.6900 | 119.88 |
| **Average** | **197914.51 ± 40867.82** | **443.18 ± 47.58** | **208.91 ± 30.77** | **0.5874 ± 0.0917** | **126.22 ± 14.18** |


### SMAPE Distribution Accross Folds 

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|----------------------|
| <10% | 8.9% ± 7.1% | 133 |
| 10-20% | 1.7% ± 1.5% | 26 |
| 20-30% | 2.5% ± 1.6% | 37 |
| 30-40% | 3.0% ± 2.2% | 44 |
| >40% | 84.0% ± 4.3% | 1262 |

**Comparison with Baseline:**

The GRU multivariate model with exogenous features (GDP, CPI, Interest Rate) achieves an average SMAPE of 126.22% ± 14.18%, which is **53.96 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%) and **58.40 percentage points worse** than the GRU model without exogenous features (67.82% ± 16.16%). The addition of exogenous variables significantly degrades performance, with 84.0% of series exhibiting SMAPE >40% compared to 54.7% in the non-exogenous version. The model shows poor R² scores (0.5874 ± 0.0917) and high error metrics (RMSE: 443.18), suggesting that the exogenous features introduce noise rather than useful signal for these time series. This counterintuitive result indicates that the economic indicators (GDP, CPI, Interest Rate) may not have direct predictive power for these specific series, or the model architecture struggles to effectively integrate the additional features, leading to overfitting or confounding the temporal patterns learned from the historical values alone.
